# 08. Statistical Significance Testing (RQ1-3)

This notebook evaluates the statistical significance of token fertility reductions achieved by **GhanaTok** (Byte-Level BPE & SentencePiece Unigram) compared against general multilingual baselines (**mBERT**, **XLM-R / AfroXLMR**) and regional baselines (**ABENA**) on held-out test sets (608 Twi sentences, 347 Ewe sentences).

### Methodology
1. **Paired Comparisons (RQ2)**: Every tokenizer is benchmarked on identical sentences in identical order. We use the non-parametric **Wilcoxon signed-rank test** for p-values and **1,000-iteration paired bootstrap** for 95% confidence intervals on absolute differences and percentage fertility reductions.
2. **Multiple Comparisons Correction**: **Holm-Bonferroni step-down correction** is applied across the primary family of 5 comparisons.
3. **Cross-Language Comparison (RQ3)**: Independent sample comparison (Twi vs. Ewe) using the **Mann-Whitney U test** and unpaired bootstrap confidence intervals.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure repo root is on sys.path
sys.path.insert(0, str(Path.cwd().parent))
from src.significance import (
    get_paired_fertility,
    paired_wilcoxon_test,
    paired_bootstrap_ci,
    unpaired_bootstrap_ci,
    apply_holm_bonferroni
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

## 1. Load Held-Out Benchmark Results

In [ ]:
results_path = Path("../reports/tables/final_heldout_results.csv")
df = pd.read_csv(results_path)
print(f"Loaded {len(df)} sentence evaluation rows.")
df.head()

## 2. RQ2 Significance Results (Primary & Bonus Comparisons)

In [ ]:
rq2_table_path = Path("../reports/tables/rq2_significance_results.csv")
rq2_df = pd.read_csv(rq2_table_path)
rq2_df

## 3. RQ3 Unpaired Cross-Language Comparison (Twi vs Ewe)

In [ ]:
rq3_table_path = Path("../reports/tables/rq3_significance_results.csv")
rq3_df = pd.read_csv(rq3_table_path)
rq3_df

## 4. Visualizing Paired Differences and Bootstrap Distributions

In [ ]:
twi_ghana, twi_mbert = get_paired_fertility(df, "twi", "GhanaTok", "mBERT")
ewe_ghana, ewe_mbert = get_paired_fertility(df, "ewe", "GhanaTok", "mBERT")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Twi differences
sns.histplot(twi_ghana - twi_mbert, kde=True, ax=axes[0], color="#2b5c8f", bins=25)
axes[0].axvline(0, color="red", linestyle="--", label="Zero Diff (Null)")
axes[0].axvline((twi_ghana - twi_mbert).mean(), color="black", linestyle="-", label=f"Mean Diff: {(twi_ghana - twi_mbert).mean():.3f}")
axes[0].set_title("Twi: Sentence-Level Fertility Diff (GhanaTok - mBERT)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Fertility Difference (tokens/word)")
axes[0].legend()

# Ewe differences
sns.histplot(ewe_ghana - ewe_mbert, kde=True, ax=axes[1], color="#e27d60", bins=25)
axes[1].axvline(0, color="red", linestyle="--", label="Zero Diff (Null)")
axes[1].axvline((ewe_ghana - ewe_mbert).mean(), color="black", linestyle="-", label=f"Mean Diff: {(ewe_ghana - ewe_mbert).mean():.3f}")
axes[1].set_title("Ewe: Sentence-Level Fertility Diff (GhanaTok - mBERT)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Fertility Difference (tokens/word)")
axes[1].legend()

plt.tight_layout()
plt.show()